In [ ]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation Completed")
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("done")
GROQ_API_KEY="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"
os.environ["GROq_API_KEY"]=GROQ_API_KEY

groq_client=Groq(api_key=GROQ_API_KEY)
print("Done")

Installation Completed
done
Done


In [ ]:
df=pd.read_csv('college_notes.csv')
print(df.shape)

print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))

print('\n Length of content')
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].head(3).to_string(index=False))


(14, 4)
Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    1
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N01

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata=[{
    'subject':row['subject'],
    'topic':row['topic']
}for row in df.to_dict('records')]

print(f'Total chunks prepared:{len(documents)}')
print(f'First document ID:{ids[1]}')
print(f'First document metadata:{metadata[1]}')
print(f'First 100 chars of document:{documents[1][:100]}')

Total chunks prepared:14
First document ID:note_N002
First document metadata:{'subject': 'Data Engineering', 'topic': 'SQL Databases'}
First 100 chars of document:A database is an organized collection of data stored electronically. SQL or Structured Query Languag


In [ ]:
chroma_client=chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("done")
print(f'Dcoument count{collection.count()}')

done
Dcoument count14


In [ ]:
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f'Shape of embeddings:{embeddings.shape}')
embeddings=embeddings.tolist()
print(f'First 5 embeddings:{embeddings[0][:5]}')

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadata,
    ids=ids
)

print(f'\nDocuments successfully added')
print(f'Document count:{collection.count()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape of embeddings:(14, 384)
First 5 embeddings:[-0.06544743478298187, 0.04776368290185928, 0.0007756710983812809, 0.0279158353805542, 0.027123192325234413]

Documents successfully added
Document count:14


In [ ]:
student_question = input("Please enter your question: ")
print(f"You asked: {student_question}")

query_embedding = embedding_model.encode([student_question]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    include=['documents', 'metadatas', 'distances']
)

print("\nTop 3 most relevant notes:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n--- Result {i+1} ---")
    print(f"Topic: {meta['topic']}")
    print(f"Subject: {meta['subject']}")
    print(f"Content: {doc[:200]}...")

Please enter your question: What is ETL?
You asked: What is ETL?

Top 3 most relevant notes:

--- Result 1 ---
Topic: ETL Pipelines
Subject: Data Engineering
Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehou...

--- Result 2 ---
Topic: APIs and Data Collection
Subject: Data Engineering
Content: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock price...

--- Result 3 ---
Topic: Retrieval Augmented Generation
Subject: Generative AI
Content: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This re...
